In [1]:
import numpy as np
from ioMicro import *
from scipy.spatial import KDTree

In [2]:
all_flds = glob.glob(r'Z:\Adam\E201_WTC11_WTday15__10_20_2023\H*_RMER_Q*')
all_flds = np.array(all_flds)[np.argsort([get_iH(fld) for fld in all_flds])]
print(len(all_flds))
save_folder = r'Z:\Adam\E201_WTC11_WTday15__10_20_2023\AnalysisDeconvolve_CG\RNA_exon'
segmentation_folder = r'Z:\Adam\E201_WTC11_WTday15__10_20_2023\AnalysisDeconvolve_CG\segmentationDAPI'
segm_fls = np.sort(glob.glob(segmentation_folder+os.sep+'*expanded.npz'))
fls = np.sort(glob.glob(all_flds[0]+os.sep+'*.zarr'))

16


In [3]:
len(segm_fls)

0

In [ ]:
XposA,XcellsA = None,None
volsA,ifovsA = None,None
cell_idsA = None
for ifl in tqdm(np.arange(len(segm_fls))):
    fl = segm_fls[ifl]
    imseg = np.load(fl)['segm']
    shape_ =  np.load(fl)['shape']
    pix_size = shape_/imseg.shape*[0.3,0.108333,0.108333]
    cell_ids,vols = np.unique(imseg,return_counts=True)
    vols = vols[cell_ids>0]
    cell_ids = cell_ids[cell_ids>0]
    from scipy import ndimage as ndi
    Xcells = np.array(ndi.center_of_mass(imseg>0,imseg,cell_ids))
    Xcells = Xcells*pix_size
    im_fl = fls[ifl]
    Xpos = read_im(im_fl,return_pos=True)[1:]
    Xpos = np.array([[0]+list(Xpos)]*len(Xcells))
    XposA = Xpos if XposA is None else np.concatenate([XposA,Xpos])
    XcellsA = Xcells if XcellsA is None else np.concatenate([XcellsA,Xcells])
    volsA =  vols if volsA is None else np.concatenate([volsA,vols])
    ifovs = np.array([ifl]*len(Xcells))
    ifovsA =  ifovs if ifovsA is None else np.concatenate([ifovsA,ifovs])
    cell_idsA =  cell_ids if cell_idsA is None else np.concatenate([cell_idsA,cell_ids])

In [ ]:
cell_idsA = None
for ifl in tqdm(np.arange(len(segm_fls))):
    fl = segm_fls[ifl]
    imseg = np.load(fl)['segm']
    cell_ids = np.unique(imseg)
    cell_ids = cell_ids[cell_ids>0]
    cell_idsA =  cell_ids if cell_idsA is None else np.concatenate([cell_idsA,cell_ids])

In [ ]:
Xf = XposA*[1,-1,-1]+XcellsA
#np.percentile(volsA,25)
ddist,ineigh = KDTree(Xf).query(Xf,10)
ifovM = ifovsA[ineigh]
keep = ifovM!=ifovM[:,0][:,np.newaxis]
keep&=ddist<10
i1,i2 = np.where(keep)
i2 = ineigh[i1,i2]
remove = np.unique(np.concatenate([i1[volsA[i1]<volsA[i2]],i2[volsA[i2]<volsA[i1]]]))
keep = np.setdiff1d(np.arange(len(Xf)),remove)

In [ ]:
np.savez(r'Z:\Adam\E201_WTC11_WTday15__10_20_2023\AnalysisDeconvolve_CG\RNA_exon\cell_meta_statistics.npz',
         Xpos=XposA,Xcells=XcellsA,ifovs=ifovsA,vols=volsA,Xf=XposA*[1,-1,-1]+XcellsA,keep=keep,cell_ids=cell_idsA)

In [6]:
dic = np.load(r'Z:\Adam\E201_WTC11_WTday15__10_20_2023\AnalysisDeconvolve_CG\RNA_exon\cell_meta_statistics.npz')
Xf,vols,ifovs,cell_ids = dic['Xf'],dic['vols'],dic['ifovs'],dic['cell_ids']
cell_idsf = [str(ifov)+'_'+str(cell_id) for ifov,cell_id in zip(ifovs,cell_ids)]
cell_idsf = np.array(cell_idsf)
cell_to_i = {c:i for i,c in enumerate(cell_idsf)}
gns_names = ['ABHD17A', 'ACHE', 'ADAM11', 'ADAMTSL5', 'ADAT3', 'ADCK5', 'AGAP6', 'AIF1L', 'ALDH1B1', 'ALDOC', 'ALK', 'ANKRD18A', 'ANKRD24', 'ANO5', 'ANTXR2', 'APBA3', 'APC2', 'ARHGAP39', 'ATCAY', 'ATP5F1D', 'ATP8B3', 'ATP9A', 'AURKA', 'BLCAP', 'BOP1', 'BTBD2', 'C10orf53', 'C19orf25', 'C1orf53', 'C20orf194', 'C5orf47', 'C8orf82', 'CACNG4', 'CALD1', 'CAPN6', 'CAPS2', 'CBARP', 'CCDC179', 'CCNA2', 'CCNB2', 'CCNE1', 'CCNE2', 'CCNF', 'CD22', 'CDH3', 'CDK1', 'CEP19', 'CFAP299', 'CHAT', 'CHGB', 'CHRNA3', 'CIRBP', 'CLIP4', 'CNN2', 'CNR1', 'CNTN1', 'CPEB4', 'CPSF1', 'CREB3L3', 'CSMD3', 'CSNK1G2', 'CTNNA2', 'CTNNBL1', 'CTSC', 'CYC1', 'CYP11B2', 'DAPK3', 'DAZAP1', 'DCX', 'DENND1B', 'DGAT1', 'DMKN', 'DNMT3B', 'DPPA4', 'DRGX', 'DSG2', 'EBI3', 'ECT2', 'EEF1D', 'EEF2', 'EFNA2', 'EPB41L1', 'EPCAM', 'ERCC6', 'EXOSC4', 'FAM174C', 'FAM187B', 'FANCF', 'FBXL6', 'FBXO45', 'FFAR1', 'FFAR2', 'FFAR3', 'FGF5', 'FLT1', 'FOXH1', 'FOXN1', 'FXYD1', 'FXYD3', 'FXYD5', 'FXYD7', 'GAMT', 'GAS2', 'GFPT2', 'GFUS', 'GHRH', 'GLI4', 'GLIPR1', 'GLIPR1L1', 'GLIPR1L2', 'GML', 'GNAO1', 'GPAA1', 'GPIHBP1', 'GPR42', 'GPT', 'GRAMD1A', 'GSDMD', 'HAMP', 'HEY1', 'HGH1', 'HPN', 'HSF1', 'IDS', 'IFT20', 'IGFBPL1', 'ITGB8', 'KCNC2', 'KIAA0100', 'KIFAP3', 'KIFC2', 'KLF16', 'KRR1', 'KRTDAP', 'LGI4', 'LHX9', 'LRRC14', 'LRRC24', 'LRRC7', 'LSR', 'LY6E', 'LY6H', 'LY6L', 'LYRM9', 'MACC1', 'MAF1', 'MAFA', 'MAG', 'MANBAL', 'MAP2', 'MAP2K2', 'MAPK8IP2', 'MAPRE3', 'MATK', 'MBD3', 'MCM3', 'MCM6', 'MEX3D', 'MFSD3', 'MIDN', 'MROH1', 'MROH6', 'MROH8', 'MRPL54', 'MRPS28', 'NAPRT', 'NAV3', 'NCBP2', 'NCBP2AS2', 'NDUFS7', 'NEK7', 'NLK', 'NMRK2', 'NNAT', 'NOS2', 'NRROS', 'NSG2', 'NUSAP1', 'OGDHL', 'ONECUT3', 'ORC1', 'PAK2', 'PAK3', 'PARG', 'PCARE', 'PCBP4', 'PCSK4', 'PIAS4', 'PIGS', 'PIGX', 'PIGZ', 'PIP5K1C', 'PITPNM2', 'PLK5', 'POLDIP2', 'POU4F2', 'PPP1R16A', 'PPP2R5B', 'PRDM8', 'PSD', 'PTPRN', 'PWWP3A', 'PYCR3', 'RAX2', 'RBL1', 'RECQL4', 'REEP1', 'REEP6', 'REXO1', 'RHPN1', 'RNF168', 'RPN2', 'RPS15', 'RSKR', 'SALL4', 'SARM1', 'SBSN', 'SCAMP4', 'SCN1B', 'SCN2A', 'SCN3A', 'SCRT1', 'SCX', 'SEBOX', 'SENP5', 'SEPHS1', 'SERTM2', 'SHARPIN', 'SHB', 'SIRT6', 'SLC10A7', 'SLC13A2', 'SLC17A6', 'SLC18A3', 'SLC38A11', 'SLC39A14', 'SLC39A4', 'SLC46A1', 'SLC4A8', 'SLC52A2', 'SMCO1', 'SNAP91', 'SPACA1', 'SPAG5', 'SPP1', 'SRC', 'STMN2', 'SVIP', 'SYN1', 'SYT13', 'TCF3', 'TIGD5', 'TIMM23B', 'TIMM23B-AGAP6', 'TJP3', 'TMEM199', 'TMEM249', 'TMEM97', 'TMPO', 'TNFAIP1', 'TOGARAM2', 'TONSL', 'TOP1MT', 'TPD52', 'TTC29', 'UBXN7', 'UNC119', 'UQCR11', 'USF2', 'USP28', 'VPS28', 'VTN', 'WASHC2A', 'WDR43', 'WDR53', 'WDR97', 'WEE1', 'YBX3', 'ZBTB7A', 'ZC3H3', 'ZFP41', 'ZFR2', 'ZNF30', 'ZNF623', 'ZNF696', 'ZNF792', 'ZNF804A', 'blank0001', 'blank0002', 'blank0003', 'blank0004', 'blank0005', 'blank0006', 'blank0007', 'blank0008', 'blank0009', 'blank0010', 'blank0011', 'blank0012', 'blank0013', 'blank0014', 'blank0015', 'blank0016', 'blank0017', 'blank0018', 'blank0019', 'blank0020', 'blank0021']

In [7]:
Mcts = np.zeros([len(cell_idsf),len(gns_names)],dtype=int)

In [4]:
fls_RNA = np.sort(glob.glob(r'Z:\Adam\E201_WTC11_WTday15__10_20_2023\AnalysisDeconvolve_CG\best_AdamExonsNew\*.npz'))
len(fls_RNA)

0

In [9]:
len(comb_ids_u) == len(cell_ids_)
print(len(cell_idsf))
print(len(keep))
XH.shape

NameError: name 'comb_ids_u' is not defined

In [10]:
for ifov in tqdm(np.arange(len(fls_RNA))):
    fl = fls_RNA[ifov]
    XH = np.load(fl)['XH_fs']
    cell_ids_ = XH[:,0,-1]
    XH = XH[cell_ids_>0]
    gene_ids_ = XH[:,0,-2]
    cell_ids_ = XH[:,0,-1]
    nmax = np.max(gene_ids_)+1
    comb_ids_ = nmax*cell_ids_+gene_ids_
    comb_ids_u, ncts_comb = np.unique(comb_ids_,return_counts=True)
    comb_ids_u=comb_ids_u
    cell_ids_u = (comb_ids_u//nmax).astype(int)
    gene_ids_u = (comb_ids_u%nmax).astype(int)
    cell_ids_ = [str(ifov)+'_'+str(c) for c in cell_ids_u]
    cellI=[]
    keep = []
    for c in cell_ids_:
        if c in cell_to_i.keys():
            keep.append(True)
            cellI.append(cell_to_i[c])
        else:
            keep.append(False)
            continue
    keep = np.array(keep)
    #cellI = [cell_to_i[c] for c in cell_ids_]]
    Mcts[cellI,gene_ids_u[keep]]=ncts_comb[keep]

100%|█████████████████████████████████████████████████████████████████████████████| 308/308 [04:25<00:00,  1.16it/s]


In [115]:
Mcts_copy = Mcts.copy()

In [116]:
keep  = np.intersect1d(dic['keep'],np.where(vols>2000)[0])

In [11]:
cell_idsf[keep]

IndexError: boolean index did not match indexed array along dimension 0; dimension is 64124 but corresponding boolean dimension is 17571

In [117]:
np.savez(r'Z:\Adam\E201_WTC11_WTday15__10_20_2023\AnalysisDeconvolve_CG\RNA_exon\count_matrixExons.npz',
         cell_idsf=cell_idsf[keep],Xf=Xf,vols=vols,Mcts=Mcts,keep = keep,gns_names=gns_names)